In [6]:
import sys
sys.path.append('../Common')

import CommonBinance

In [7]:
import talib 

def detectSignal(symbol, from_date, to_date, timeframe):

    import pandas as pd
    import plotly.graph_objects as go
    import redis
    import numpy as np
    from datetime import datetime
    from statsmodels.tsa.arima.model import ARIMA

    # ##############################################Step 1: Lấy dữ liệu##############################################
    data = CommonBinance.CommonBinance.loaddataBinance_FromTo(symbol, from_date, to_date, timeframe)
    
    # ##############################################Step 2: Chiến lược##############################################  
    # Giả sử `data` là DataFrame chứa các cột Datetime, Open, High, Low, Close, Volume
    # Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
    data.set_index('Datetime', inplace=True)

    # Tính Momentum
    data['Momentum'] = talib.MOM(data['Close'], timeperiod=14)

    # Tính RSI
    data['RSI'] = talib.RSI(data['Close'], timeperiod=14)

    # Chọn tham số ARIMA dựa trên AIC, BIC hoặc các tiêu chí khác (cần phải thực hiện trước) => Chay cell o tren
    model = ARIMA(data['Close'], order=(5, 1, 5))
    model_fit = model.fit()

    # Dự đoán giá
    data['Predicted_Close'] = model_fit.predict(start=0, end=len(data)-1, typ='levels')

    # Xác định tín hiệu mua và bán
    data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['Momentum'] > -10) & (data['RSI'] < 170))
    data['Sell_Signal'] = ((data['Predicted_Close'] < data['Close']) & (data['Momentum'] < 2000) & (data['RSI'] > 30))

    # ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
    # Nếu có tín hiệu thì đẩy qua Redis
    # Datetime  Open    High    Low	Close   Volume  SMA short_ema   long_ema    MACD    Signal_Line Buy_Signal  Sell_Signal
    # Tạo kết nối Redis
    r = redis.Redis(host='localhost', port=6379, db=15)
    # Tạo hash key
    hash_key = 'OG_CRTO_Arima, Momentum, RSI_' + symbol # OG_CRTOFT_Arima, Momentum, RSI
    last_record = data.iloc[-2]
    print(last_record)
    # Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
    if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
        for field, value in last_record.to_dict().items():
            # Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
            if isinstance(value, pd.Timestamp):
                value = value.isoformat()
            elif isinstance(value, (int, np.uint64)):
                value = str(value)
            r.hset(hash_key, field, value)  
            r.hset(hash_key, 'Symbol', symbol)
            r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
        print(last_record)   
    else:
        print('Không có tín hiệu!')   

In [ ]:
from datetime i             ddddddd datetime, timedelta
import schedule
import timeNEARf schedule_detectSignal():
    symbol = 'NEARUSDT'
    from_date = (datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
    to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
    timeframe = '1m' # 1m
    detectSignal(symbol, from_date, to_date, timeframe)

In [9]:
# run_minutes = list(range(0,60,1)) # Chay tung 1 phut tu phut 0 den phut 59

# while True:
#     current_time = datetime.now()
#     current_minute = current_time.minute
#     # Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
#     if current_minute in run_minutes:
#         # Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
#         last_run = getattr(schedule_detectSignal, 'last_run', None)
#         if last_run is None or last_run != current_minute:
#             schedule_detectSignal() # Chay o giay 0
#             # Lưu lại phút cuối cùng mà hàm đã chạy
#             setattr(schedule_detectSignal, 'last_run', current_minute)   

#     # Nghỉ 1 giây trước khi kiểm tra lại
#     time.sleep(1)

In [10]:
import schedule
# Moi phut chay 1 lan tai giay thu 0
schedule.every().minute.at(":00").do(schedule_detectSignal)
while True:
    schedule.run_pending()
    time.sleep(1)

c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
c:\Users\TGC\App

Open                   2.211
High                   2.212
Low                    2.208
Close                  2.209
Volume                5192.2
Momentum               -0.01
RSI                37.993523
Predicted_Close        2.212
Buy_Signal              True
Sell_Signal            False
Name: 2025-10-22 14:20:00, dtype: object
Open                   2.211
High                   2.212
Low                    2.208
Close                  2.209
Volume                5192.2
Momentum               -0.01
RSI                37.993523
Predicted_Close        2.212
Buy_Signal              True
Sell_Signal            False
Name: 2025-10-22 14:20:00, dtype: object


c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
c:\Users\TGC\AppData\Local\Progra

Open                   2.208
High                   2.209
Low                    2.203
Close                  2.205
Volume               35765.4
Momentum              -0.018
RSI                34.363996
Predicted_Close     2.208091
Buy_Signal              True
Sell_Signal            False
Name: 2025-10-22 14:21:00, dtype: object
Open                   2.208
High                   2.209
Low                    2.203
Close                  2.205
Volume               35765.4
Momentum              -0.018
RSI                34.363996
Predicted_Close     2.208091
Buy_Signal              True
Sell_Signal            False
Name: 2025-10-22 14:21:00, dtype: object


c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
c:\Users\TGC\AppData\Local\Progra

...

c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)


Open                   2.195
High                   2.196
Low                    2.189
Close                  2.193
Volume               11826.2
Momentum              -0.012
RSI                42.560965
Predicted_Close     2.193767
Buy_Signal              True
Sell_Signal            False
Name: 2025-10-22 14:35:00, dtype: object
Open                   2.195
High                   2.196
Low                    2.189
Close                  2.193
Volume               11826.2
Momentum              -0.012
RSI                42.560965
Predicted_Close     2.193767
Buy_Signal              True
Sell_Signal            False
Name: 2025-10-22 14:35:00, dtype: object


c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
c:\Users\TGC\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Open                   2.195
High                   2.196
Low                    2.189
Close                  2.193
Volume               11826.2
Momentum              -0.012
RSI                42.560965
Predicted_Close     2.193733
Buy_Signal              True
Sell_Signal            False
Name: 2025-10-22 14:35:00, dtype: object
Open                   2.195
High                   2.196
Low                    2.189
Close                  2.193
Volume               11826.2
Momentum              -0.012
RSI                42.560965
Predicted_Close     2.193733
Buy_Signal              True
Sell_Signal            False
Name: 2025-10-22 14:35:00, dtype: object


RequestTimeout: binance GET https://fapi.binance.com/fapi/v1/exchangeInfo